In [1]:
import requests

url = "https://www.gutenberg.org/cache/epub/22403/pg22403-images.html"
headers = {"User-Agent": "INFO664-class-project (educational)"}

response = requests.get(url, headers=headers)

with open("gutenberg.html", "w", encoding="utf-8") as file:
    file.write(response.text)

In [2]:
import csv
import json # allows us to write json files and interpret requests as JSON
import os # operating system library, let's us do things like making new folders
from time import sleep
from bs4 import BeautifulSoup # the scraping library!

In [3]:
SCRAPED_PAGES_DIRECTORY = "scraped_pages/"
SCRAPED_PAGES_PREFIX = "page"
scraped_pages_count = 0 # keep track of how many pages we have scraped

headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    global scraped_pages_count  # use the scraped_pages_count from the top of the script, instead of making a new local one
    filename = f"{SCRAPED_PAGES_DIRECTORY}{SCRAPED_PAGES_PREFIX}{scraped_pages_count}.html"
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        scraped_pages_count = scraped_pages_count + 1
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)
        scraped_pages_count = scraped_pages_count + 1
    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

In [9]:
url = "https://www.gutenberg.org/cache/epub/22403/pg22403-images.html"
html = fetch_page(url) # filling the var "html" with the actual html, as text
soup = BeautifulSoup(html, "html.parser") # new var, converting the text into something structured using the Beautiful Soup library

In [10]:
poem_titles = []
title_element = soup.find_all(name="i")
title_elements_filtered = []
for title in title_element:
    if title.get_text(strip=True)[0].isdigit():
        poem_title = "".join(filter(lambda c: c.isalpha() or c.isspace(), title.get_text())) # Filter out non-letter characters, found this on stack overflow
        poem_titles.append(poem_title)
        title_elements_filtered.append(title)
print(poem_titles)

['', '', '\r\n\r\nFor a Picture of\r\n\r\nSt Dorothea', '\r\n\r\nHeavenHaven\r\n\r\nA nun takes the veil', '\r\nThe Habit of Perfection', '\r\n\r\nTHE WRECK\r\n\r\nOF THE DEUTSCHLAND', '\r\nPenmaen Pool', '\r\n\r\nThe Silver Jubilee\r\n\r\nTo James First Bishop of Shrewsbury on the th Year\r\n\r\nof his Episcopate July  ', '\r\nGods Grandeur', '\r\nThe Starlight Night', '\r\nSpring', '\r\nThe Lantern out of Doors', '\r\nThe Sea and the Skylark', '\r\nPied Beauty', '\r\nHurrahing in Harvest', '\r\nCaged Skylark', '\r\nIn the Valley of the Elwy', '\r\nThe May Magnificat', '\r\nDuns Scotuss Oxford', '\r\nHenry Purcell', '\r\nPeace', '\r\nMorning Midday and Evening Sacrifice', '\r\nAndromeda', '\r\nThe Candle Indoors', '\r\nFelix Randal', '\r\nBrothers', '\r\nSpelt from Sibyls Leaves', '\r\nInversnaid', '\r\nRibblesdale', '\r\n\r\nThe Blessed Virgin compared to the Air we\r\n\r\nBreathe', '\r\nTo what serves Mortal Beauty', '\r\nThe Soldier', '\r\nCarrion Comfort', '', '\r\nHarry Ploughman

In [11]:
poem_titles = []
title_element = soup.find_all(name="i")
title_elements_filtered = []
for title in title_element:
    if title.get_text(strip=True)[0].isdigit():
        poem_title = "".join(filter(lambda c: c.isalpha() or c.isspace(), title.get_text())) # Filter out non-letter characters, found this on stack overflow
        poem_titles.append(poem_title)
        title_elements_filtered.append(title)
print(poem_titles)

['', '', '\r\n\r\nFor a Picture of\r\n\r\nSt Dorothea', '\r\n\r\nHeavenHaven\r\n\r\nA nun takes the veil', '\r\nThe Habit of Perfection', '\r\n\r\nTHE WRECK\r\n\r\nOF THE DEUTSCHLAND', '\r\nPenmaen Pool', '\r\n\r\nThe Silver Jubilee\r\n\r\nTo James First Bishop of Shrewsbury on the th Year\r\n\r\nof his Episcopate July  ', '\r\nGods Grandeur', '\r\nThe Starlight Night', '\r\nSpring', '\r\nThe Lantern out of Doors', '\r\nThe Sea and the Skylark', '\r\nPied Beauty', '\r\nHurrahing in Harvest', '\r\nCaged Skylark', '\r\nIn the Valley of the Elwy', '\r\nThe May Magnificat', '\r\nDuns Scotuss Oxford', '\r\nHenry Purcell', '\r\nPeace', '\r\nMorning Midday and Evening Sacrifice', '\r\nAndromeda', '\r\nThe Candle Indoors', '\r\nFelix Randal', '\r\nBrothers', '\r\nSpelt from Sibyls Leaves', '\r\nInversnaid', '\r\nRibblesdale', '\r\n\r\nThe Blessed Virgin compared to the Air we\r\n\r\nBreathe', '\r\nTo what serves Mortal Beauty', '\r\nThe Soldier', '\r\nCarrion Comfort', '', '\r\nHarry Ploughman

In [53]:
# craig working out ideas here

poems = [] # empty list to store poem-dictionaries
# the below for loop structure generates TWO interated variables – the current title (title_el) and the next title (next_title_el)
# using the zip() function https://www.w3schools.com/python/ref_func_zip.asp
for title_el, next_title_el in zip(title_elements_filtered, title_elements_filtered[1:]):
    current_title_parent = title_el.find_parent() # find the paragraph element since that's the sibling we're looking for
    next_title_parent = next_title_el.find_parent() # find the paragraph element of the next title so we can look for a match...
    
    if next_title_el is not None: 
        next_title_parent = next_title_el.find_parent()
    else: # handling the case in which there is NO next title (end of doc)
        next_title_parent = None
    poem = [] # poem list to store all the individual stanzas
    current_el = current_title_parent # the element we are currently finding the next sibling of (which we will change in the loop)
    while True: # loop until break
        next_sibling = current_el.find_next_sibling() # find the next p element
        
        # STOP if we hit the next title's parent, OR if we run out of siblings completely
        if next_sibling is next_title_parent or next_sibling is None:
            break
        
        poem.append(next_sibling.get_text()) # if we are here, it means we found another p element which contains poem text, store it
        joined_poem = "".join(poem)
        current_el = next_sibling  # Move forward
    # build a dictionary with the title and the stanzas in "poem_text" 
    poem_object = {
        "title": title_el.get_text(),
        "poem_text": joined_poem,
    }
    poems.append(poem_object) # add that dictionary to the list of all poems
    
print(len(poems)) #shows us how many poems were stored

# print(poems[5]) # if we print it renders super weird cuz of all the non-printing characters. instead let's use pandas, it renders the text more cleanly

70


In [54]:
import pandas as pd

df = pd.DataFrame(poems)

In [55]:
print(df["poem_text"][3])

   I HAVE desired to go

     Where springs not fail,

To fields where flies no sharp and sided hail

   And a few lilies blow.
   And I have asked to be

     Where no storms come,

Where the green swell is in the havens dumb,

   And out of the swing of the sea.

